# Sleep Cannabis Alcohol Study

- Merging merged actigraphy data from the 4 studies with the scored (scrubbed) actigraphy data from each study

Notes: 
- Use scored data for start and stop times.
- Use the actigraphy data for determining sleep metrics
- Calculate sleep biomarkers for: 
  1. sleep onset to sleep offset (lets me verify data outputs compared to orignal scored dataset)
  2. First 1/2 of sleep night i.e., start sleep to mid-sleep)
  3. first 4-h of sleep (sleep start to sleep start + 4h)

## Load Data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

#### Actigraphy Data

In [4]:
actigraphy_df = pd.read_csv('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/Data_analysis/full_actigraphy_df_cleaned_09.10.26.csv', index_col=0, dtype={'date':str, 'time':str})

actigraphy_df.head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
0,DXA_174,DXA,3/27/2025,12:49:00 PM,769,2025-03-27 12:49:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
1,DXA_174,DXA,3/27/2025,12:50:00 PM,770,2025-03-27 12:50:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
2,DXA_174,DXA,3/27/2025,12:51:00 PM,771,2025-03-27 12:51:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
3,DXA_174,DXA,3/27/2025,12:52:00 PM,772,2025-03-27 12:52:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
4,DXA_174,DXA,3/27/2025,12:53:00 PM,773,2025-03-27 12:53:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah


In [5]:
actigraphy_df[actigraphy_df['subject_id']=='siesta2_218'].head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
827677,siesta2_218,Siesta2,10/21/2024,8:28:00 PM,1228,2024-10-21 20:28:00,0,149.0,0.0,NaN,ACTIVE,si218_10_21_2024_8_28_00_pm_si218_24hr_ks
827678,siesta2_218,Siesta2,10/21/2024,8:29:00 PM,1229,2024-10-21 20:29:00,0,514.0,0.0,NaN,ACTIVE,si218_10_21_2024_8_28_00_pm_si218_24hr_ks
827679,siesta2_218,Siesta2,10/21/2024,8:30:00 PM,1230,2024-10-21 20:30:00,0,298.0,0.0,1.0,ACTIVE,si218_10_21_2024_8_28_00_pm_si218_24hr_ks
827680,siesta2_218,Siesta2,10/21/2024,8:31:00 PM,1231,2024-10-21 20:31:00,0,339.0,0.0,1.0,ACTIVE,si218_10_21_2024_8_28_00_pm_si218_24hr_ks
827681,siesta2_218,Siesta2,10/21/2024,8:32:00 PM,1232,2024-10-21 20:32:00,0,339.0,0.0,1.0,ACTIVE,si218_10_21_2024_8_28_00_pm_si218_24hr_ks


In [21]:
## convert dtypes for analysis
actigraphy_df['subject_id'] = actigraphy_df['subject_id'].astype(str)
actigraphy_df['subject_id'] = actigraphy_df['subject_id'].replace('SNA1', 'SNA').replace('SPU1', 'SPU')
actigraphy_df['date_time24'] = pd.to_datetime(actigraphy_df['date'] + ' ' + actigraphy_df['time'], format='mixed')

actigraphy_df.head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
0,DXA_174,DXA,3/27/2025,12:49:00 PM,769,2025-03-27 12:49:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
1,DXA_174,DXA,3/27/2025,12:50:00 PM,770,2025-03-27 12:50:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
2,DXA_174,DXA,3/27/2025,12:51:00 PM,771,2025-03-27 12:51:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
3,DXA_174,DXA,3/27/2025,12:52:00 PM,772,2025-03-27 12:52:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
4,DXA_174,DXA,3/27/2025,12:53:00 PM,773,2025-03-27 12:53:00,0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah


In [6]:
actigraphy_df[actigraphy_df['subject_id']=='DXA_021'].head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
437873,DXA_021,DXA,10/30/2019,2:49:00 PM,889,2019-10-30 14:49:00,0,159.0,0.0,NaN,ACTIVE,dxa021_10_30_2019_2_49_00_pm_cc_24hr_combined_ks
437874,DXA_021,DXA,10/30/2019,2:50:00 PM,890,2019-10-30 14:50:00,0,6.0,0.0,NaN,ACTIVE,dxa021_10_30_2019_2_49_00_pm_cc_24hr_combined_ks
437875,DXA_021,DXA,10/30/2019,2:51:00 PM,891,2019-10-30 14:51:00,0,17.0,0.0,1.0,ACTIVE,dxa021_10_30_2019_2_49_00_pm_cc_24hr_combined_ks
437876,DXA_021,DXA,10/30/2019,2:52:00 PM,892,2019-10-30 14:52:00,0,97.0,0.0,1.0,ACTIVE,dxa021_10_30_2019_2_49_00_pm_cc_24hr_combined_ks
437877,DXA_021,DXA,10/30/2019,2:53:00 PM,893,2019-10-30 14:53:00,0,182.0,0.0,1.0,ACTIVE,dxa021_10_30_2019_2_49_00_pm_cc_24hr_combined_ks


In [7]:
actigraphy_df['subject_id'].nunique()

256

### Import passcode-subject_id code list

- this is used to map passcodes and subject_ids from both the epoch-by-epoch actigraph files and the old/new scored sleep summary data dfs

In [22]:
passcodes_df = pd.read_excel('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/scored_summary_actigraphy_data/passcode_study_df_091626.xlsx', index_col=0)

passcodes_df['subject_id'] = passcodes_df['subject_id'].replace(r'1_','_', regex=True)
# passcodes_df['subject_id'] = passcodes_df['subject_id'].replace('SNA1', 'SNA').replace('SPU1', 'SPU')


passcodes_df.head()

,subject_id,passcode,study
0,SNA_154801,Lynn,SNA
1,SNA_154804,Bergen,SNA
2,SNA_154808,Serra,SNA
3,SNA_154809,Salavan,SNA
4,SNA_154810,Preston,SNA


In [23]:
passcodes_df['subject_id'].nunique()

259

In [24]:
actigraphy_df['subject_id'].nunique()

256

In [ ]:
## see what subject_id values from passcodes are not in (merged) actigraphy_df 
passcodes_no_actigraphy = passcodes_df[~passcodes_df['subject_id'].isin(actigraphy_df['subject_id'].values)]
## see what subject_id values from actigraphy_df are not in the passcodes list
actigraphy_no_passcodes = actigraphy_df[~actigraphy_df['subject_id'].isin(passcodes_df['subject_id'].values)]

print(f'{passcodes_no_actigraphy['subject_id'].values} are not in actigraphy files')
print(f'{actigraphy_no_passcodes['subject_id'].values} have actigraph files but are not in passcodes')

## DXA_005 has 6 of 13 nights of sleep data (excluded)
## DXA_105 has 5 of 6 nights of sleep data collected (excluded for insufficient # of nights)
## DXA_128 has 6 of 7 nights of sleep data collected (excluded for insufficient # of nights)

['DXA_005' 'DXA_105' 'DXA_128'] are not in actigraphy files
[] have actigraph files but are not in passcodes


In [26]:
# subject_ids in actigraphy_df but missing from passcodes_df
missing_from_passcodes = (
    actigraphy_df.loc[
        ~actigraphy_df['subject_id'].isin(passcodes_df['subject_id']),
        'subject_id'
    ]
    .unique()
)

# subject_ids in passcodes_df but missing from actigraphy_df
missing_from_actigraphy = (
    passcodes_df.loc[
        ~passcodes_df['subject_id'].isin(actigraphy_df['subject_id']),
        'subject_id'
    ]
    .unique()
)

print("Missing from passcodes_df:", missing_from_passcodes)
print("Missing from actigraphy_df:", missing_from_actigraphy)

Missing from passcodes_df: []
Missing from actigraphy_df: ['DXA_005' 'DXA_105' 'DXA_128']


### Revised Scored Sleep Data

In [20]:
scored_test = pd.read_excel('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/scored_summary_actigraphy_data/AllStudies_Diary_Actigraphy_SleepTiming_ALL.xlsx')

scored_test.columns = scored_test.columns.str.lower()

scored_test.head()

,subjectid,passcode,studyday,start_date,start_day,start_time,end_date,end_day,end_time,start_date_nondst,...,midpoint_timing,midpoint_time,date,night,timetosleep,sleepquality,sleepinessscale,alcohol,marijuana,alcohol_amount
0,1,DXA001,1,2019-09-20,Fri,23:21:00,2019-09-21,Sat,07:56:00,2019-09-20,...,218.5,03:38:30,2019-09-20,Friday,20.0,2.0,7.0,0.0,0.0,NaN
1,1,DXA001,2,NaT,NaN,NaN,NaT,NaN,NaN,NaT,...,NaN,NaN,2019-09-21,Saturday,10.0,4.0,3.0,0.0,0.0,NaN
2,1,DXA001,3,2019-09-22,Sun,21:42:00,2019-09-23,Mon,08:30:00,2019-09-22,...,186.0,03:06:00,2019-09-22,Sunday,5.0,2.0,6.0,0.0,0.0,NaN
3,1,DXA001,4,2019-09-23,Mon,22:21:00,2019-09-24,Tue,07:49:00,2019-09-23,...,185.0,03:05:00,2019-09-23,Monday,40.0,2.0,7.0,0.0,0.0,NaN
4,1,DXA001,5,2019-09-25,Wed,00:29:00,2019-09-25,Wed,08:20:00,2019-09-25,...,264.5,04:24:30,2019-09-24,Tuesday,30.0,1.0,4.0,0.0,0.0,NaN


#### Scored Sleep Data

In [63]:
scored_sleep_df = pd.read_excel('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/Data_analysis/sleep_merged_df_cleaned_062326.xlsx', index_col=0)
scored_sleep_df['subject_id'] = scored_sleep_df['subject_id'].astype(str)
scored_sleep_df['subject_id'] = scored_sleep_df['subject_id'].astype(str)
scored_sleep_df['subject_id'] = scored_sleep_df['subject_id'].str.replace('SNA1', 'SNA').str.replace('SPU1', 'SPU')

dt_columns = ['start_date', 'start_time', 'end_date',
       'end_time', 'start_datetime', 'end_datetime', 'sleep_mid_dt', 'sleep_onset_plus4h']

for col in dt_columns:
    scored_sleep_df[col] = pd.to_datetime(scored_sleep_df[col])

scored_sleep_df.head()

/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_33845/3161584176.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scored_sleep_df[col] = pd.to_datetime(scored_sleep_df[col])
/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_33845/3161584176.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scored_sleep_df[col] = pd.to_datetime(scored_sleep_df[col])


,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-09-10 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
1,DXA_001,DXA001,Sleep,2,Saturday,NaT,NaN,NaT,NaT,NaN,...,NaN,EXCLUDED - >15% of the day off-wrist,DXA,NaN,NaT,NaT,NaN,NaN,NaT,NaT
2,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-09-10 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
3,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-09-10 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
4,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-09-10 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00


In [64]:
# scored_sleep_df[scored_sleep_df['subject_id']=='DXA_021'][['flag_actigraph']]

scored_sleep_df[scored_sleep_df['subject_id']=='DXA_021'][['subject_id', 'flag_actigraph']]


,subject_id,flag_actigraph
202,DXA_021,DST Night: 11/3/19 at 2:00 AM;
203,DXA_021,Times adjsuted 1 hour behind for DST
204,DXA_021,Times adjsuted 1 hour behind for DST
205,DXA_021,Times adjsuted 1 hour behind for DST
206,DXA_021,Times adjsuted 1 hour behind for DST
207,DXA_021,Times adjsuted 1 hour behind for DST
208,DXA_021,Times adjsuted 1 hour behind for DST
209,DXA_021,Times adjsuted 1 hour behind for DST
210,DXA_021,Times adjsuted 1 hour behind for DST
211,DXA_021,Times adjsuted 1 hour behind for DST


## Data Cleaning

In [65]:
actigraphy_IDS = list(actigraphy_df['subject_id'].unique())

scored_IDs = list(scored_sleep_df['subject_id'].unique())

In [66]:
missing_in_scored = set(actigraphy_IDS) - set(scored_IDs)

print(f'Participants in Actigraphy but not in scored dataset:')
print(f'{missing_in_scored}')

Participants in Actigraphy but not in scored dataset:
set()


In [67]:
missing_in_actigraphy = set(scored_IDs) - set(actigraphy_IDS)

print('Participants in scored dataset not found in the actigraphy_files_df:' )
print(f'{missing_in_actigraphy}')

Participants in scored dataset not found in the actigraphy_files_df:
{'DXA_105', 'DXA_128', 'DXA_005'}


## Filter participants to drop

In [68]:
## drop these participants based on their status from completing the study (i.e., withdrew or D/c for one reason or another)
pid_to_drop = list(missing_in_actigraphy)

pid_to_drop

['DXA_105', 'DXA_128', 'DXA_005']

## Drop excluded nights of sleep 

In [69]:
## Filter out flagged nights from scored_sleep_df
print("flag_actigraph value counts:")
print(scored_sleep_df['flag_actigraph'].value_counts(dropna=False))

flag_actigraph value counts:
flag_actigraph
NaN                                                                                                                                                                                  2585
EXCLUDED - >15% of the day off-wrist                                                                                                                                                  142
Times adjsuted 1 hour behind for DST                                                                                                                                                  105
DST - adjusted sleep onset and offset to 1 hr back                                                                                                                                     74
OFF-WRIST - cannot determine sleep onset or offset                                                                                                                                     38
ALL-NIGHTER               

# Start of Problem section

In [70]:
exclude_terms = ['EXCLUDE', 'REMOVE', 'DROP']   ## add terms for filtering from flag_actigraph values

scored_sleep_filtered_df = scored_sleep_df[~scored_sleep_df['flag_actigraph'].str.contains(
        '|'.join(exclude_terms),
        case=False,
        na=False,
        regex=True
    )].copy().reset_index(drop=True)

print(f"Total nights before flag filter : {len(scored_sleep_df)}")
print(f"Total nights after flag filter  : {len(scored_sleep_filtered_df)}")
print(f"Nights dropped                  : {len(scored_sleep_df) - len(scored_sleep_filtered_df)}")

Total nights before flag filter : 3062
Total nights after flag filter  : 2885
Nights dropped                  : 177


In [71]:
## keep only nights where flag_actigraph is null/empty (not flagged)
## This is the original Python cell but it needs fixing as null/empty is not correct. Delete after confirming correction works.

# scored_sleep_filtered_df = scored_sleep_df[scored_sleep_df['flag_actigraph'].isna()].copy()

# print(f"Total nights before flag filter : {len(scored_sleep_df)}")
# print(f"Total nights after flag filter  : {len(scored_sleep_filtered_df)}")
# print(f"Nights dropped                  : {len(scored_sleep_df) - len(scored_sleep_filtered_df)}")

In [72]:
scored_sleep_filtered_df.shape

(2885, 38)

In [73]:
## keep only actigraphy data from participants that did not withdraw from study 
valid_scored_df = scored_sleep_filtered_df[~scored_sleep_filtered_df['subject_id'].isin(pid_to_drop)].copy()

print(f"Total nights before flag filter : {len(scored_sleep_filtered_df)}")
print(f"Total nights after flag filter  : {len(valid_scored_df)}")
print(f"Nights dropped                  : {len(scored_sleep_filtered_df) - len(valid_scored_df)}")

Total nights before flag filter : 2885
Total nights after flag filter  : 2866
Nights dropped                  : 19


In [74]:
valid_scored_df['subject_id'].nunique()
## 249 unique participants
# scored_sleep_filtered_df['subject_id'].nunique()
# ## 258
# scored_sleep_df['subject_id'].nunique()
## 259

256

In [75]:
valid_scored_df.head()

,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-09-10 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
1,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-09-10 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
2,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-09-10 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
3,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-09-10 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00
4,DXA_001,DXA001,Sleep,6,Wednesday,2019-09-25,Wed,2026-09-10 23:31:00,2019-09-26,Thu,...,27.33,NaN,DXA,NaN,2019-09-25 23:31:00,2019-09-26 06:59:00,0.311111,448.0,2019-09-26 03:15:00,2019-09-26 03:31:00


In [76]:
valid_scored_df[valid_scored_df['study']=='DXA']['subject_id'].unique()

array(['DXA_001', 'DXA_002', 'DXA_003', 'DXA_004', 'DXA_008', 'DXA_010',
       'DXA_011', 'DXA_013', 'DXA_014', 'DXA_015', 'DXA_017', 'DXA_020',
       'DXA_021', 'DXA_022', 'DXA_023', 'DXA_025', 'DXA_026', 'DXA_029',
       'DXA_032', 'DXA_033', 'DXA_036', 'DXA_037', 'DXA_038', 'DXA_039',
       'DXA_040', 'DXA_043', 'DXA_045', 'DXA_046', 'DXA_048', 'DXA_049',
       'DXA_052', 'DXA_053', 'DXA_054', 'DXA_055', 'DXA_056', 'DXA_058',
       'DXA_064', 'DXA_066', 'DXA_067', 'DXA_068', 'DXA_069', 'DXA_070',
       'DXA_075', 'DXA_080', 'DXA_082', 'DXA_083', 'DXA_086', 'DXA_090',
       'DXA_092', 'DXA_097', 'DXA_098', 'DXA_099', 'DXA_100', 'DXA_101',
       'DXA_103', 'DXA_108', 'DXA_109', 'DXA_110', 'DXA_112', 'DXA_113',
       'DXA_114', 'DXA_118', 'DXA_129', 'DXA_130', 'DXA_140', 'DXA_142',
       'DXA_148', 'DXA_156', 'DXA_158', 'DXA_159', 'DXA_162', 'DXA_164',
       'DXA_168', 'DXA_169', 'DXA_170', 'DXA_171', 'DXA_173', 'DXA_174',
       'DXA_176', 'DXA_177', 'DXA_178', 'DXA_180'],

In [77]:
## Standardize datetimes and confirm they're in datetime format
valid_scored_df['start_datetime'] = pd.to_datetime(valid_scored_df['start_datetime'])
valid_scored_df['sleep_mid_dt']   = pd.to_datetime(valid_scored_df['sleep_mid_dt'])
valid_scored_df['sleep_onset_plus4h'] = pd.to_datetime(valid_scored_df['sleep_onset_plus4h'])
valid_scored_df['end_datetime'] = pd.to_datetime(valid_scored_df['end_datetime'])
## actigraphy_df pd.to_datime
actigraphy_df['date_time24']      = pd.to_datetime(actigraphy_df['date_time24'])

## doublecheck
print("Dtypes confirmed:")
print(valid_scored_df[['start_datetime', 'sleep_mid_dt', 'sleep_onset_plus4h', 'end_datetime']].dtypes)
print(actigraphy_df['date_time24'].dtype)

Dtypes confirmed:
start_datetime        datetime64[ns]
sleep_mid_dt          datetime64[ns]
sleep_onset_plus4h    datetime64[ns]
end_datetime          datetime64[ns]
dtype: object
datetime64[ns]


# End of Problem section

## Calculate Fragmentation Index

Fragmentation Index = 

((Mobile Epochs + Immobile Bouts ≤1 min) / Total Immobile Bouts) * 100

#### Vanilla_SFI

In [78]:
## Vanilla Fragmentation Index Calculation 7.9.26
## A = mobile epochs (activity count >= 40)
## B = sandwiched immobile epochs (activity < 40, but both pre- proceeding epochs >=40 ac)
## C = total immobile epochs (ac <40)

def vanilla_fragmentation_index(activity_series, activity_threshold=40):
    """
    Compute Fragmentation index from series of activity counts > the sleep period (sleep start to sleep end)
    returns fragmentation index as a percentage, plus A, B, C, components
    fragmentation index = ((A+B)/C)*100
    """
    act = activity_series.reset_index(drop=True)

    mobile_mask = act >= activity_threshold
    A = int(mobile_mask.sum())

    immobile_mask = act < activity_threshold
    C = int(immobile_mask.sum())

    B = 0
    for i in range(1,len(act)-1):
        if (
            act[i] < activity_threshold and
            act[i-1] >= activity_threshold and
            act[i+1] >= activity_threshold
        ):
            B +=1
            
    fragmentation_index = round(((A+B)/C)*100, 2) if C >0 else None

    return {
        'A_mobile_epochs_van': A,
        'B_sandwiched_immobile_van': B,
        'C_total_immobile_van' : C,
        'vanilla_SFI': fragmentation_index,
    }

#### Saleh SFI

In [79]:
## Saleh et al. (2025) Sleep Fragmentation Index (SFI) calculation
from itertools import groupby

def saleh_fragmentation_index(activity_series, activity_threshold=2):
    """
    Compute Saleh et al. (2025) Sleep Fragmentation Index (SFI)
    over the sleep period (sleep onset to sleep offset / TIB).

    saleh_SFI = A_prime + B_prime

    A_prime = (A / B) * 100
        A = # epochs with AC >= threshold (mobile epochs)
        B = # total epochs between sleep onset and offset (TIB)

    B_prime = (C / D) * 100
        C = # immobile bouts <= 2 epochs in length
        D = # total immobile bouts (any length)

    An immobile bout = consecutive epochs where AC < threshold.

    Parameters
    ----------
    activity_series    : pd.Series of raw activity counts (sleep onset to offset)
    activity_threshold : int, default 2
    """
    act = activity_series.reset_index(drop=True)

    # A: mobile epochs (AC >= threshold)
    A = int((act >= activity_threshold).sum())

    # B: total epochs in window (TIB)
    B = len(act)

    # identify all immobile bouts using run-length encoding
    immobile_bouts = [
        list(group)
        for key, group in groupby(act, lambda x: x < activity_threshold)
        if key  # only immobile runs
    ]

    ## D: total number of immobile bouts
    D = len(immobile_bouts)

    ## C: immobile bouts that are <= 2 epochs in length
    C = sum(1 for bout in immobile_bouts if len(bout) <= 2)

    ## guard against division by zero
    A_prime = round((A / B) * 100, 2) if B > 0 else None
    B_prime = round((C / D) * 100, 2) if D > 0 else None
    SFI     = round(A_prime + B_prime, 2) if (A_prime is not None and B_prime is not None) else None

    return {
        'A_mobile_epochs_saleh'       : A,
        'B_TIB_epochs_saleh'          : B,
        'C_2consec_immobile_bouts_saleh': C,
        'D_total_immobile_bouts_saleh': D,
        'A_prime_saleh'               : A_prime,
        'B_prime_saleh'               : B_prime,
        'saleh_SFI'          : SFI,
    }

#### Actiware and Actilife SFI calculations
(Liu SFI Calculation)

- SFI formula #1 (ref [21], Respironics inc., Actiware user manual; 2013)

**Actiware_Respironics_SFI** = A_Prime +B_Prime

A_Prime = (A/B)*100
B_Prime = (C/D)*100

A = # epochs with AC ≥ 2 

B = # epochs between sleep_start and sleep_end (i.e., TIB) 

C = # bouts with 2 consecutive epochs where AC <2 

D = # Bouts with consecutive epochs where AC <2 

Notes: 

"SFI was expressed as the sum of two percentages as shown below [above] with higher values indicates more resltess sleep [21]:"

**ActiLife_SFI:** = C_Prime / D_Prime

C_Prime = (A/B)*100
D_Prime = (C/D)*100

A = # epochs with AC > 0 on the vertical axis 

B = # Epochs during TIB (sleep_start to sleep_end) 

C = # of sleep bouts of 1-min length (aka "sandwiched" sleep intervals) 

D = # of sleep bouts (aka total of "immobile" epochs, or TST)

Notes: 

"Comparing equations for SFI used by Actiware and ActiLife, the difference is that ActiLife calculates the second percentage based on sleep/wake status whereas the Actiware is based on activity counts. There was no easy fix to make this parameter comparable as we did for the other parameters. However, we retained th eparameter in ur analyses because the two equations—despite the different operationalization—appear to be designed to estimate the same underlying construct..."



In [80]:
from itertools import groupby

def Liu_fragmentation_index(activity_series, actiware_ac_threshold=2, actilife_ac_threshold=0):
    """
    Compute Liu et al. (2024) Sleep Fragmentation Index (SFI)

    ======== Actiware_SFI =========
    Actiware_SFI = A_Prime + B_Prime (%)

    A_prime = (A / B) * 100
        A = # epochs with AC >= actiware_ac_threshold (mobile epochs)
        B = # total epochs between sleep onset and offset (TIB)

    B_prime = (C / D) * 100
        C = # immobile bouts <= 2 epochs in length
        D = # total immobile bouts

    ======== ActiLife_SFI =========
    ActiLife_SFI = C_Prime + D_Prime (%)

    C_Prime = (E / F) * 100
        E = # epochs with AC > actilife_ac_threshold (mobile epochs)
        F = # total epochs during TIB

    D_Prime = (G / H) * 100
        G = # sandwiched immobile epochs (1-min bouts)
        H = # total immobile bouts
    """
    act = activity_series.reset_index(drop=True)

    ######## Actiware_SFI formula ########

    # A: mobile epochs (AC >= actiware threshold)
    A = int((act >= actiware_ac_threshold).sum())

    # B: total epochs in window (TIB)
    B = len(act)

    # identify all immobile bouts via run-length encoding
    immobile_bouts = [
        list(group)
        for key, group in groupby(act, lambda x: x < actiware_ac_threshold)
        if key
    ]

    # D: total number of immobile bouts
    D = len(immobile_bouts)

    # C: immobile bouts <= 2 epochs in length
    C = sum(1 for bout in immobile_bouts if len(bout) <= 2)

    # guard against division by zero
    A_prime      = round((A / B) * 100, 2) if B > 0 else None
    B_prime      = round((C / D) * 100, 2) if D > 0 else None
    Actiware_SFI = round(A_prime + B_prime, 2) if (A_prime is not None and B_prime is not None) else None

    ######## ActiLife_SFI formula ########

    # E: mobile epochs (AC > actilife threshold)
    E = int((act > actilife_ac_threshold).sum())

    # F: total epochs in window (TIB)
    F = len(act)

    # G: sandwiched immobile epochs (single immobile epoch between two mobile epochs)
    G = 0
    for i in range(1, len(act) - 1):
        if (
            act[i]     <= actilife_ac_threshold and
            act[i - 1] >  actilife_ac_threshold and
            act[i + 1] >  actilife_ac_threshold
        ):
            G += 1

    # H: total immobile bouts via run-length encoding
    actilife_immobile_bouts = [
        list(group)
        for key, group in groupby(act, lambda x: x <= actilife_ac_threshold)
        if key
    ]
    H = len(actilife_immobile_bouts)

    # guard against division by zero
    C_prime      = round((E / F) * 100, 2) if F > 0 else None
    D_prime      = round((G / H) * 100, 2) if H > 0 else None
    Actilife_SFI = round(C_prime + D_prime, 2) if (C_prime is not None and D_prime is not None) else None

    return {
        'A_mobile_epochs_actiware'        : A,
        'B_TIB_epochs_actiware'           : B,
        'C_short_immobile_bouts_actiware' : C,
        'D_total_immobile_bouts_actiware' : D,
        'A_prime_actiware'                : A_prime,
        'B_prime_actiware'                : B_prime,
        'Actiware_SFI'                    : Actiware_SFI,
        'E_mobile_epochs_actilife'        : E,
        'F_TIB_epoch_actilife'            : F,
        'G_sandwiched_immobile_actilife'  : G,
        'H_total_immobile_bouts_actilife' : H,
        'C_prime_actilife'                : C_prime,
        'D_prime_actilife'                : D_prime,
        'Actilife_SFI'           : Actilife_SFI,
    }

## Test calculations

In [91]:
test_pid = ['DXA_001']
DXA_001_df = valid_scored_df[valid_scored_df['subject_id'].isin(test_pid)].copy()

print(DXA_001_df.shape)
display(DXA_001_df.head())

(13, 38)


,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-09-10 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
1,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-09-10 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
2,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-09-10 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
3,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-09-10 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00
4,DXA_001,DXA001,Sleep,6,Wednesday,2019-09-25,Wed,2026-09-10 23:31:00,2019-09-26,Thu,...,27.33,NaN,DXA,NaN,2019-09-25 23:31:00,2019-09-26 06:59:00,0.311111,448.0,2019-09-26 03:15:00,2019-09-26 03:31:00


In [92]:
## testing DXA_001's records before the larger, full actigraphy dataset.
DXA_001_records = []

for _, row in DXA_001_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['end_datetime']      

    # slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    # --- sleep/wake classification ---
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = vanilla_fragmentation_index(sleep_period_activity, activity_threshold=40)
        frag_saleh            = saleh_fragmentation_index(sleep_period_activity, activity_threshold=2)
        frag_liu               = Liu_fragmentation_index(sleep_period_activity, actiware_ac_threshold = 2, actilife_ac_threshold=0)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs_van'      : None,
            'B_sandwiched_immobile_van': None,
            'C_total_immobile_van'     : None,
            'vanilla_SFI'              : None,
        }
        frag_saleh = {
        'A_mobile_epochs_saleh'       : None,
        'B_TIB_epochs_saleh'          : None,
        'C_2consec_immobile_bouts_saleh': None,
        'D_total_immobile_bouts_saleh': None,
        'A_prime_saleh'               : None,
        'B_prime_saleh'               : None,
        'saleh_SFI'                   : None,
        }
        liu_SFI = {
        'A_mobile_epochs_actiware'        : None,
        'B_TIB_epochs_actiware'           : None,
        'C_short_immobile_bouts_actiware' : None,
        'D_total_immobile_bouts_actiware' : None,
        'A_prime_actiware'                : None,
        'B_prime_actiware'                : None,
        'Actiware_SFI'                    : None,
        'E_mobile_epochs_actilife'        : None,
        'F_TIB_epochs_actilife'           : None,
        'G_sandwiched_immobile_actilife'  : None,
        'H_total_immobile_bouts_actilife' : None,
        'C_prime_actilife'                : None,
        'D_prime_actilife'                : None,
        'ActiLife_SFI'                    : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    DXA_001_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
        **frag_saleh,
        **frag_liu
    })

DXA_001_sleep_results_df = pd.DataFrame(DXA_001_records)
print(f"Results shape: {DXA_001_sleep_results_df.shape}")

DXA_001_sleep_results_df.head(5)

Results shape: (13, 34)


,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,24,...,8.35,12.12,20.47,43,515,4,33,8.35,12.12,20.47
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,36,...,10.80,18.75,29.55,70,648,5,48,10.80,10.42,21.22
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,43,...,14.96,39.13,54.09,85,568,12,46,14.96,26.09,41.05
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,13,...,9.77,26.32,36.09,46,471,7,38,9.77,18.42,28.19
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,28,...,10.27,22.22,32.49,46,448,8,36,10.27,22.22,32.49


In [93]:
SFI_filtered_cols = ['subject_id', 'interval#', 'start_datetime', 'end_datetime',
      'vanilla_SFI', 'saleh_SFI', 'Actiware_SFI','Actilife_SFI']

In [94]:
DXA_001_sleep_results_df[SFI_filtered_cols]

,subject_id,interval#,start_datetime,end_datetime,vanilla_SFI,saleh_SFI,Actiware_SFI,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,5.50,20.47,20.47,20.47
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,5.88,29.55,29.55,21.22
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,9.14,54.09,54.09,41.05
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,2.84,36.09,36.09,28.19
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,6.90,32.49,32.49,32.49
5,DXA_001,10,2019-09-30 00:16:00,2019-09-30 08:06:00,5.84,35.51,35.51,18.84
6,DXA_001,11,2019-09-30 23:37:00,2019-10-01 07:18:00,6.47,18.09,18.09,12.54
7,DXA_001,12,2019-10-02 00:32:00,2019-10-02 07:59:00,5.18,29.99,29.99,18.22
8,DXA_001,13,2019-10-03 00:34:00,2019-10-03 06:29:00,5.95,28.81,28.81,19.13
9,DXA_001,14,2019-10-04 00:11:00,2019-10-04 07:21:00,4.37,28.19,28.19,15.29


In [95]:
DXA_001_sleep_results_df

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,24,...,8.35,12.12,20.47,43,515,4,33,8.35,12.12,20.47
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,36,...,10.80,18.75,29.55,70,648,5,48,10.80,10.42,21.22
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,43,...,14.96,39.13,54.09,85,568,12,46,14.96,26.09,41.05
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,13,...,9.77,26.32,36.09,46,471,7,38,9.77,18.42,28.19
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,28,...,10.27,22.22,32.49,46,448,8,36,10.27,22.22,32.49
5,DXA_001,10,2019-09-30 00:16:00,2019-09-30 08:06:00,470,438,32,32,93.19,25,...,11.70,23.81,35.51,55,470,3,42,11.70,7.14,18.84
6,DXA_001,11,2019-09-30 23:37:00,2019-10-01 07:18:00,461,429,32,32,93.06,28,...,9.76,8.33,18.09,45,461,1,36,9.76,2.78,12.54
7,DXA_001,12,2019-10-02 00:32:00,2019-10-02 07:59:00,447,414,33,33,92.62,22,...,9.40,20.59,29.99,42,447,3,34,9.40,8.82,18.22
8,DXA_001,13,2019-10-03 00:34:00,2019-10-03 06:29:00,355,333,22,22,93.80,19,...,12.68,16.13,28.81,45,355,2,31,12.68,6.45,19.13
9,DXA_001,14,2019-10-04 00:11:00,2019-10-04 07:21:00,430,411,19,19,95.58,18,...,8.84,19.35,28.19,38,430,2,31,8.84,6.45,15.29


In [96]:
DXA_001_sleep_results_df.columns

Index(['subject_id', 'interval#', 'start_datetime', 'end_datetime',
       'window_min', 'TST_min', 'WASO_min', 'micro_wake_min',
       'sleep_efficiency', 'A_mobile_epochs_van', 'B_sandwiched_immobile_van',
       'C_total_immobile_van', 'vanilla_SFI', 'A_mobile_epochs_saleh',
       'B_TIB_epochs_saleh', 'C_2consec_immobile_bouts_saleh',
       'D_total_immobile_bouts_saleh', 'A_prime_saleh', 'B_prime_saleh',
       'saleh_SFI', 'A_mobile_epochs_actiware', 'B_TIB_epochs_actiware',
       'C_short_immobile_bouts_actiware', 'D_total_immobile_bouts_actiware',
       'A_prime_actiware', 'B_prime_actiware', 'Actiware_SFI',
       'E_mobile_epochs_actilife', 'F_TIB_epoch_actilife',
       'G_sandwiched_immobile_actilife', 'H_total_immobile_bouts_actilife',
       'C_prime_actilife', 'D_prime_actilife', 'Actilife_SFI'],
      dtype='object')

#### diagnostics for verifying this worked properly

In [97]:
# # ── Diagnostic 1: confirm subject_id format matches in both dfs ───────────────
# print("=== scored df subject_id for DXA_001 ===")
# print(DXA_001_df['subject_id'].unique())

# print("\n=== actigraphy_df subject_id values (all unique) ===")
# print(actigraphy_df['subject_id'].unique())

# # direct check — does DXA_001's ID appear in actigraphy_df at all?
# scored_id = DXA_001_df['subject_id'].iloc[0]
# print(f"\nLooking for '{scored_id}' in actigraphy_df...")
# print(f"Match found: {scored_id in actigraphy_df['subject_id'].values}")

In [98]:
# # ── Diagnostic 2: check datetime ranges overlap ───────────────────────────────

# actig_001 = actigraphy_df[actigraphy_df['subject_id'] == scored_id]

# print(f"=== actigraphy_df epochs for {scored_id} ===")
# print(f"row count  : {len(actig_001):,}")
# print(f"date range : {actig_001['date_time24'].min()}  →  {actig_001['date_time24'].max()}")

# print(f"\n=== DXA_001_df sleep windows ===")
# print(DXA_001_df[['interval#', 'start_datetime', 'end_datetime']].to_string(index=False))

# # check each night individually
# print("\n=== epoch count per scored night ===")
# for _, row in DXA_001_df.iterrows():
#     mask = (
#         (actigraphy_df['subject_id']  == scored_id)          &
#         (actigraphy_df['date_time24'] >= row['start_datetime']) &
#         (actigraphy_df['date_time24'] <  row['end_datetime'])
#     )
#     n = mask.sum()
#     print(f"  interval# {row['interval#']}  →  {row['start_datetime']}  to  {row['end_datetime']}  →  {n} epochs")

In [99]:
# # ── Diagnostic 3: check interval_status values for DXA_001 epochs ────────────

# print(f"=== interval_status value counts for {scored_id} in actigraphy_df ===")
# print(actig_001['interval_status'].value_counts(dropna=False))

# print(f"\n=== sleep/wake value counts ===")
# print(actig_001['sleep/wake'].value_counts(dropna=False))

# print(f"\n=== sample of raw epochs ===")
# print(actig_001[['date_time24', 'interval_status', 'sleep/wake', 'activity']].head(10).to_string(index=False))

# Sleep Onset to Sleep_offset

In [101]:
valid_scored_df['subject_id'].nunique()

256

In [ ]:
## Calculating sleep metrics + SFI versions for full actigraphy dataset.
full_noc_sleep_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['end_datetime']      

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = vanilla_fragmentation_index(sleep_period_activity, activity_threshold=40)
        frag_saleh            = saleh_fragmentation_index(sleep_period_activity, activity_threshold=2)
        frag_liu              = Liu_fragmentation_index(sleep_period_activity, actiware_ac_threshold = 2, actilife_ac_threshold=0)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs_van'      : None,
            'B_sandwiched_immobile_van': None,
            'C_total_immobile_van'     : None,
            'vanilla_SFI'              : None,
        }
        frag_saleh = {
        'A_mobile_epochs_saleh'       : None,
        'B_TIB_epochs_saleh'          : None,
        'C_2consec_immobile_bouts_saleh': None,
        'D_total_immobile_bouts_saleh': None,
        'A_prime_saleh'               : None,
        'B_prime_saleh'               : None,
        'saleh_SFI'                   : None,
        }
        liu_SFI = {
        'A_mobile_epochs_actiware'        : None,
        'B_TIB_epochs_actiware'           : None,
        'C_short_immobile_bouts_actiware' : None,
        'D_total_immobile_bouts_actiware' : None,
        'A_prime_actiware'                : None,
        'B_prime_actiware'                : None,
        'Actiware_SFI'                    : None,
        'E_mobile_epochs_actilife'        : None,
        'F_TIB_epochs_actilife'           : None,
        'G_sandwiched_immobile_actilife'  : None,
        'H_total_immobile_bouts_actilife' : None,
        'C_prime_actilife'                : None,
        'D_prime_actilife'                : None,
        'ActiLife_SFI'                    : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    full_noc_sleep_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
        **frag_saleh,
        **frag_liu
    })

full_noc_sleep_results_df = pd.DataFrame(full_noc_sleep_records)
print(f"Results shape: {full_noc_sleep_results_df.shape}")

Results shape: (2525, 34)


In [ ]:
display(full_noc_sleep_results_df.head())
display(full_noc_sleep_results_df.tail())

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,24.0,...,8.35,12.12,20.47,43,515,4,33,8.35,12.12,20.47
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,36.0,...,10.80,18.75,29.55,70,648,5,48,10.80,10.42,21.22
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,43.0,...,14.96,39.13,54.09,85,568,12,46,14.96,26.09,41.05
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,13.0,...,9.77,26.32,36.09,46,471,7,38,9.77,18.42,28.19
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,28.0,...,10.27,22.22,32.49,46,448,8,36,10.27,22.22,32.49


,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
2520,SPU_154418,3,2024-03-28 01:07:00,2024-03-28 10:09:00,542,445,97,97,82.10,81.0,...,21.96,36.62,58.58,119,542,19,71,21.96,26.76,48.72
2521,SPU_154418,4,2024-03-28 23:31:00,2024-03-29 09:26:00,595,518,77,77,87.06,64.0,...,16.97,40.68,57.65,101,595,13,59,16.97,22.03,39.00
2522,SPU_154418,5,2024-03-30 01:10:00,2024-03-30 08:37:00,447,397,50,50,88.81,38.0,...,16.78,23.68,40.46,75,447,4,38,16.78,10.53,27.31
2523,SPU_154418,6,2024-03-30 22:02:00,2024-03-31 08:30:00,628,463,165,165,73.73,123.0,...,35.83,51.39,87.22,225,628,18,72,35.83,25.00,60.83
2524,SPU_154418,7,2024-04-01 00:50:00,2024-04-01 08:24:00,454,410,44,44,90.31,33.0,...,14.10,30.95,45.05,64,454,6,42,14.10,14.29,28.39


In [ ]:
full_SFI_df = pd.merge(full_noc_sleep_results_df, valid_scored_df[['subject_id', 'interval#', 'fragmentation']],
                  on = ['subject_id', 'interval#'],
                  how ='left' 
                  )
full_SFI_df = full_SFI_df.rename(columns={'fragmentation':'scored_fragmentation'})

SFI_cols = list(['subject_id', 'interval#', 'start_datetime', 'end_datetime','vanilla_SFI', 'saleh_SFI', 'Actiware_SFI', 'Actilife_SFI', 'scored_fragmentation'])

full_SFI_df[SFI_cols].tail()

,subject_id,interval#,start_datetime,end_datetime,vanilla_SFI,saleh_SFI,Actiware_SFI,Actilife_SFI,scored_fragmentation
2520,SPU_154418,3,2024-03-28 01:07:00,2024-03-28 10:09:00,20.17,58.58,58.58,48.72,45.10
2521,SPU_154418,4,2024-03-28 23:31:00,2024-03-29 09:26:00,13.37,57.65,57.65,39.00,35.63
2522,SPU_154418,5,2024-03-30 01:10:00,2024-03-30 08:37:00,10.27,40.46,40.46,27.31,25.29
2523,SPU_154418,6,2024-03-30 22:02:00,2024-03-31 08:30:00,27.72,87.22,87.22,60.83,58.64
2524,SPU_154418,7,2024-04-01 00:50:00,2024-04-01 08:24:00,7.84,45.05,45.05,28.39,25.19


In [ ]:
full_sleep_filtered_cols_to_keep = ['subject_id',
 'interval#',
 'start_datetime',
 'end_datetime',
 'window_min',
 'TST_min',
 'WASO_min',
 'micro_wake_min',
 'sleep_efficiency',
 'vanilla_SFI',
 'saleh_SFI',
 'Actiware_SFI',
 'Actilife_SFI',
 'scored_fragmentation'
 ]

In [ ]:
full_SFI_df[full_sleep_filtered_cols_to_keep].head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,vanilla_SFI,saleh_SFI,Actiware_SFI,Actilife_SFI,scored_fragmentation
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,5.50,20.47,20.47,20.47,20.28
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,5.88,29.55,29.55,21.22,20.75
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,9.14,54.09,54.09,41.05,28.54
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,2.84,36.09,36.09,28.19,17.80
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,6.90,32.49,32.49,32.49,27.33


## Export Full_sleep_results


In [ ]:
full_SFI_df[full_sleep_filtered_cols_to_keep].to_csv('full_noc_sleep_results_df_091026.csv')

# Sleep Metrics: sleep onset to mid-sleep

In [ ]:
valid_scored_df.columns

Index(['subject_id', 'passcode', 'interval_type', 'interval#', 'night',
       'start_date', 'start_day', 'start_time', 'end_date', 'end_day',
       'end_time', 'duration', 'off_wrist', 'pcn_off_wrist', 'totalac',
       'avg_ac/min', 'max_ac', 'onsetlat', 'eff', 'waso', 'waketime', '%wake',
       '#wakebouts', 'avg_wake_b', 'sleep_time', '%_sleep', '#sleepbouts',
       'avgsleepb', 'fragmentation', 'flag_actigraph', 'study', 'semester',
       'start_datetime', 'end_datetime', 'time_diff', 'tst_min',
       'sleep_mid_dt', 'sleep_onset_plus4h'],
      dtype='object')

In [ ]:
## sleep onset to mid-sleep sleep biometrics and SFI calculations 
onset_mid_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['sleep_mid_dt']      

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = vanilla_fragmentation_index(sleep_period_activity, activity_threshold=40)
        frag_saleh            = saleh_fragmentation_index(sleep_period_activity, activity_threshold=2)
        frag_liu              = Liu_fragmentation_index(sleep_period_activity, actiware_ac_threshold = 2, actilife_ac_threshold=0)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs_van'      : None,
            'B_sandwiched_immobile_van': None,
            'C_total_immobile_van'     : None,
            'vanilla_SFI'              : None,
        }
        frag_saleh = {
        'A_mobile_epochs_saleh'       : None,
        'B_TIB_epochs_saleh'          : None,
        'C_2consec_immobile_bouts_saleh': None,
        'D_total_immobile_bouts_saleh': None,
        'A_prime_saleh'               : None,
        'B_prime_saleh'               : None,
        'saleh_SFI'                   : None,
        }
        liu_SFI = {
        'A_mobile_epochs_actiware'        : None,
        'B_TIB_epochs_actiware'           : None,
        'C_short_immobile_bouts_actiware' : None,
        'D_total_immobile_bouts_actiware' : None,
        'A_prime_actiware'                : None,
        'B_prime_actiware'                : None,
        'Actiware_SFI'                    : None,
        'E_mobile_epochs_actilife'        : None,
        'F_TIB_epochs_actilife'           : None,
        'G_sandwiched_immobile_actilife'  : None,
        'H_total_immobile_bouts_actilife' : None,
        'C_prime_actilife'                : None,
        'D_prime_actilife'                : None,
        'ActiLife_SFI'                    : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    onset_mid_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
        **frag_saleh,
        **frag_liu
    })

onset_mid_sleep_df = pd.DataFrame(onset_mid_records)
print(f"Results shape: {onset_mid_sleep_df.shape}")

Results shape: (2525, 34)


In [ ]:
onset_mid_sleep_df.head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 03:38:30,258,243,15,15,94.19,12.0,...,8.14,13.33,21.47,21,258,2,15,8.14,13.33,21.47
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 03:06:00,324,305,19,19,94.14,17.0,...,12.65,25.93,38.58,41,324,5,27,12.65,18.52,31.17
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 03:05:00,284,252,32,32,88.73,25.0,...,18.66,53.85,72.51,53,284,10,26,18.66,38.46,57.12
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 04:24:30,236,232,4,4,98.31,4.0,...,6.36,14.29,20.65,15,236,1,14,6.36,7.14,13.50
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 03:15:00,224,210,14,14,93.75,13.0,...,10.27,16.67,26.94,23,224,3,18,10.27,16.67,26.94


In [ ]:
filtered_cols_to_keep = ['subject_id',
 'interval#',
 'start_datetime',
 'end_datetime',
 'window_min',
 'TST_min',
 'WASO_min',
 'micro_wake_min',
 'sleep_efficiency',
 'vanilla_SFI',
 'saleh_SFI',
 'Actiware_SFI',
 'Actilife_SFI',
 ]

In [ ]:
onset_mid_sleep_df[filtered_cols_to_keep].head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,vanilla_SFI,saleh_SFI,Actiware_SFI,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 03:38:30,258,243,15,15,94.19,5.28,21.47,21.47,21.47
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 03:06:00,324,305,19,19,94.14,5.54,38.58,38.58,31.17
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 03:05:00,284,252,32,32,88.73,11.20,72.51,72.51,57.12
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 04:24:30,236,232,4,4,98.31,1.72,20.65,20.65,13.50
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 03:15:00,224,210,14,14,93.75,6.16,26.94,26.94,26.94


In [ ]:
onset_mid_sleep_df[filtered_cols_to_keep].to_csv('Onset2mid_sleep_results_091026.csv')

# Sleep onset to sleep onset + 4h

In [ ]:
## sleep onset to plus 4h sleep results and SFI calculations 

onset_plus4h_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['sleep_onset_plus4h']      

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = vanilla_fragmentation_index(sleep_period_activity, activity_threshold=40)
        frag_saleh            = saleh_fragmentation_index(sleep_period_activity, activity_threshold=2)
        frag_liu              = Liu_fragmentation_index(sleep_period_activity, actiware_ac_threshold = 2, actilife_ac_threshold=0)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs_van'      : None,
            'B_sandwiched_immobile_van': None,
            'C_total_immobile_van'     : None,
            'vanilla_SFI'              : None,
        }
        frag_saleh = {
        'A_mobile_epochs_saleh'       : None,
        'B_TIB_epochs_saleh'          : None,
        'C_2consec_immobile_bouts_saleh': None,
        'D_total_immobile_bouts_saleh': None,
        'A_prime_saleh'               : None,
        'B_prime_saleh'               : None,
        'saleh_SFI'                   : None,
        }
        liu_SFI = {
        'A_mobile_epochs_actiware'        : None,
        'B_TIB_epochs_actiware'           : None,
        'C_short_immobile_bouts_actiware' : None,
        'D_total_immobile_bouts_actiware' : None,
        'A_prime_actiware'                : None,
        'B_prime_actiware'                : None,
        'Actiware_SFI'                    : None,
        'E_mobile_epochs_actilife'        : None,
        'F_TIB_epochs_actilife'           : None,
        'G_sandwiched_immobile_actilife'  : None,
        'H_total_immobile_bouts_actilife' : None,
        'C_prime_actilife'                : None,
        'D_prime_actilife'                : None,
        'ActiLife_SFI'                    : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    onset_plus4h_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
        **frag_saleh,
        **frag_liu
    })

onset_plus4h_sleep_df = pd.DataFrame(onset_plus4h_records)
print(f"Results shape: {onset_plus4h_sleep_df.shape}")

Results shape: (2525, 34)


In [ ]:
onset_plus4h_sleep_df.head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 03:21:00,240,225,15,15,93.75,12.0,...,8.75,20.00,28.75,21,240,2,15,8.75,13.33,22.08
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 01:42:00,240,223,17,17,92.92,15.0,...,14.58,33.33,47.91,35,240,5,21,14.58,23.81,38.39
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 02:21:00,240,210,30,30,87.50,23.0,...,20.83,60.87,81.70,50,240,9,23,20.83,39.13,59.96
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 04:29:00,240,236,4,4,98.33,4.0,...,6.25,14.29,20.54,15,240,1,14,6.25,7.14,13.39
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 03:31:00,240,226,14,14,94.17,13.0,...,10.00,15.79,25.79,24,240,3,19,10.00,15.79,25.79


In [ ]:
onset_plus4h_sleep_df[filtered_cols_to_keep].to_csv('plus_results_df_091026.csv')

# Mid-sleep to Sleep offset

In [ ]:
## Second half sleep biometrics plus SFI calculations 
second_half_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['sleep_mid_dt']
    t_end   = row['end_datetime']     

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = vanilla_fragmentation_index(sleep_period_activity, activity_threshold=40)
        frag_saleh            = saleh_fragmentation_index(sleep_period_activity, activity_threshold=2)
        frag_liu              = Liu_fragmentation_index(sleep_period_activity, actiware_ac_threshold = 2, actilife_ac_threshold=0)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs_van'      : None,
            'B_sandwiched_immobile_van': None,
            'C_total_immobile_van'     : None,
            'vanilla_SFI'              : None,
        }
        frag_saleh = {
        'A_mobile_epochs_saleh'       : None,
        'B_TIB_epochs_saleh'          : None,
        'C_2consec_immobile_bouts_saleh': None,
        'D_total_immobile_bouts_saleh': None,
        'A_prime_saleh'               : None,
        'B_prime_saleh'               : None,
        'saleh_SFI'                   : None,
        }
        liu_SFI = {
        'A_mobile_epochs_actiware'        : None,
        'B_TIB_epochs_actiware'           : None,
        'C_short_immobile_bouts_actiware' : None,
        'D_total_immobile_bouts_actiware' : None,
        'A_prime_actiware'                : None,
        'B_prime_actiware'                : None,
        'Actiware_SFI'                    : None,
        'E_mobile_epochs_actilife'        : None,
        'F_TIB_epochs_actilife'           : None,
        'G_sandwiched_immobile_actilife'  : None,
        'H_total_immobile_bouts_actilife' : None,
        'C_prime_actilife'                : None,
        'D_prime_actilife'                : None,
        'ActiLife_SFI'                    : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    second_half_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
        **frag_saleh,
        **frag_liu
    })

second_half_noc_sleep_df = pd.DataFrame(second_half_records)
print(f"Results shape: {second_half_noc_sleep_df.shape}")

Results shape: (2525, 34)


In [ ]:
second_half_noc_sleep_df.head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs_van,...,A_prime_actiware,B_prime_actiware,Actiware_SFI,E_mobile_epochs_actilife,F_TIB_epoch_actilife,G_sandwiched_immobile_actilife,H_total_immobile_bouts_actilife,C_prime_actilife,D_prime_actilife,Actilife_SFI
0,DXA_001,1,2019-09-21 03:38:30,2019-09-21 07:56:00,257,244,13,13,94.94,12.0,...,8.56,10.53,19.09,22,257,2,19,8.56,10.53,19.09
1,DXA_001,3,2019-09-23 03:06:00,2019-09-23 08:30:00,324,305,19,19,94.14,19.0,...,8.95,9.09,18.04,29,324,0,22,8.95,0.00,8.95
2,DXA_001,4,2019-09-24 03:05:00,2019-09-24 07:49:00,284,260,24,24,91.55,18.0,...,11.27,19.05,30.32,32,284,2,21,11.27,9.52,20.79
3,DXA_001,5,2019-09-25 04:24:30,2019-09-25 08:20:00,235,219,16,16,93.19,9.0,...,13.19,32.00,45.19,31,235,6,25,13.19,24.00,37.19
4,DXA_001,6,2019-09-26 03:15:00,2019-09-26 06:59:00,224,202,22,22,90.18,15.0,...,10.27,27.78,38.05,23,224,5,18,10.27,27.78,38.05


In [ ]:
second_half_noc_sleep_df[filtered_cols_to_keep].to_csv('second_half_noc_sleep_df_091026.csv')

# Stop